In [ ]:
!pip install --upgrade pyro-ppl

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt

import pyro
import pyro.distributions as dist
from pyro.infer.mcmc import MCMC, NUTS
import pyro.poutine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 94.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
from imblearn.over_sampling import SMOTE

df = pd.read_csv("/content/data/Flood Data.csv")

# Combine hazard columns
df["flood_severity"] = df[["HAZARD 1 IN 30", "HAZARD 1 IN 100", "HAZARD 1 IN 1000"]].max(axis=1)
df["flood_binary"] = (df["flood_severity"] > 0).astype(int)

numeric_cols = [
    "X", "Y", "ELEVATION", "SLOPE", "IMPERVIOUSNESS",
    "NDVI", "DISTANCE TO RIVER", "DISTANCE TO ROAD"
]
categorical_cols = [
    "LANDUSE", "SOIL TYPE", "SUBSTRATE", "BUILDING TYPE"
]

# Convert categorical to numeric codes
for col in categorical_cols:
    df[col] = df[col].astype('category').cat.codes

# Fill missing numeric
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean(numeric_only=True))

all_features = numeric_cols + categorical_cols
X = df[all_features].values
y = df["flood_binary"].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set distribution before SMOTE:")
print("Class 0:", sum(y_train == 0))
print("Class 1:", sum(y_train == 1))


Training set distribution before SMOTE:
Class 0: 50269
Class 1: 3595


In [ ]:
smote = SMOTE(sampling_strategy=0.4, random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Training set distribution after SMOTE=0.4:")
print("Class 0:", sum(y_train_bal == 0))
print("Class 1:", sum(y_train_bal == 1))

Training set distribution after SMOTE=0.4:
Class 0: 50269
Class 1: 20107


In [ ]:
num_idx = [all_features.index(col) for col in numeric_cols]
cat_idx = [all_features.index(col) for col in categorical_cols]

X_train_num = X_train_bal[:, num_idx]
X_train_cat = X_train_bal[:, cat_idx]
X_test_num  = X_test[:, num_idx]
X_test_cat  = X_test[:, cat_idx]

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled  = scaler.transform(X_test_num)

X_train_scaled = np.concatenate([X_train_num_scaled, X_train_cat], axis=1)
X_test_scaled  = np.concatenate([X_test_num_scaled,  X_test_cat], axis=1)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float).to(device)
y_train_t = torch.tensor(y_train_bal,   dtype=torch.float).reshape(-1, 1).to(device)
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float).to(device)
y_test_t  = torch.tensor(y_test,        dtype=torch.float).reshape(-1, 1).to(device)

print("Final training size:", X_train_t.shape, y_train_t.shape)
print("Test size:", X_test_t.shape, y_test_t.shape)

Final training size: torch.Size([70376, 12]) torch.Size([70376, 1])
Test size: torch.Size([13466, 12]) torch.Size([13466, 1])


In [ ]:
class FloodClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super(FloodClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

input_dim = X_train_t.shape[1]
deterministic_model = FloodClassifier(input_dim, hidden_dim=32).to(device)

In [ ]:
N_total = X_train_t.shape[0]
batch_size = 128

def bayesian_flood_class_model_mb(X, y, subsample_idx):

    X_batch = X[subsample_idx]
    y_batch = y[subsample_idx].squeeze(-1)  # shape [batch_size]

    # priorss
    w1_prior = dist.Normal(
        torch.zeros(32, input_dim, device=device),
        0.5 * torch.ones(32, input_dim, device=device)
    ).to_event(2)
    b1_prior = dist.Normal(
        torch.zeros(32, device=device),
        0.5 * torch.ones(32, device=device)
    ).to_event(1)

    w2_prior = dist.Normal(
        torch.zeros(32, 32, device=device),
        0.5 * torch.ones(32, 32, device=device)
    ).to_event(2)
    b2_prior = dist.Normal(
        torch.zeros(32, device=device),
        0.5 * torch.ones(32, device=device)
    ).to_event(1)

    w_out_prior = dist.Normal(
        torch.zeros(1, 32, device=device),
        0.5 * torch.ones(1, 32, device=device)
    ).to_event(2)
    b_out_prior = dist.Normal(
        torch.zeros(1, device=device),
        0.5 * torch.ones(1, device=device)
    ).to_event(1)

    priors = {
        "fc1.weight": w1_prior,
        "fc1.bias":   b1_prior,
        "fc2.weight": w2_prior,
        "fc2.bias":   b2_prior,
        "out.weight": w_out_prior,
        "out.bias":   b_out_prior
    }

    lifted_module = pyro.random_module("module", deterministic_model, priors)()
    logits = lifted_module(X_batch).squeeze(-1)

    #log prob
    base_dist = dist.Bernoulli(logits=logits)
    log_p = base_dist.log_prob(y_batch)  # shape [batch_size]

    # Slightly higher weight for minority class (y=1)

    weight_pos = 1.4
    mask_1 = (y_batch == 1).float()
    wlog_p = log_p + mask_1 * torch.log(torch.tensor(weight_pos, device=device))

    # Scale for mini-batch
    scale_batch = N_total / float(len(subsample_idx))

    with pyro.plate("data_plate", size=N_total, subsample=subsample_idx):
        # Use pyro.factor to incorporate the weighted log_prob
        pyro.factor("obs_factor", scale_batch * wlog_p.sum())

    return logits

In [ ]:
def model_wrapper_mb():
    subsample_idx = torch.randperm(N_total, device=device)[:batch_size]
    return bayesian_flood_class_model_mb(X_train_t, y_train_t, subsample_idx)


In [ ]:
pyro.clear_param_store()

nuts_kernel = NUTS(
    model_wrapper_mb,
    step_size=0.001,
    target_accept_prob=0.9
)

mcmc = MCMC(nuts_kernel, num_samples=400, warmup_steps=600, num_chains=1)
print("Starting mini-batch MCMC with NUTS...")
mcmc.run()
print("MCMC finished.")

posterior_samples = mcmc.get_samples()
print("Extracted posterior samples.")


Starting mini-batch MCMC with NUTS...


Warmup:   0%|          | 0/1000 [00:00, ?it/s]/usr/local/lib/python3.11/dist-packages/pyro/primitives.py:526: FutureWarning: The `random_module` primitive is deprecated, and will be removed in a future release. Use `pyro.nn.Module` to create Bayesian modules from `torch.nn.Module` instances.
  warnings.warn(
Sample: 100%|██████████| 1000/1000 [00:08, 120.71it/s, step size=6.19e-46, acc. prob=0.000]

MCMC finished.
Extracted posterior samples.


In [ ]:
def predict_class_from_samples(posterior_samples, X_data, num_draws=50):
    sample_keys = list(posterior_samples.keys())
    total_samples = posterior_samples[sample_keys[0]].shape[0]
    chosen_indices = np.random.choice(total_samples, size=num_draws, replace=False)

    all_probs = []
    for i in chosen_indices:
        draw_dict = {k: v[i] for k, v in posterior_samples.items()}
        lifted_module = pyro.random_module("module", deterministic_model, draw_dict)()
        lifted_module.to(device)
        with torch.no_grad():
            logits = lifted_module(X_data).squeeze(-1)
            probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
    return np.array(all_probs)

In [ ]:
print("Generating predictions on test data...")
y_prob_samples = predict_class_from_samples(posterior_samples, X_test_t, num_draws=50)
y_prob_mean = y_prob_samples.mean(axis=0)

y_test_np = y_test_t.cpu().numpy().ravel()

# ROC / PR
fpr, tpr, _ = roc_curve(y_test_np, y_prob_mean)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_test_np, y_prob_mean)
pr_auc = auc(recall, precision)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"PR AUC: {pr_auc:.4f}")


Generating predictions on test data...
ROC AUC: 0.5971
PR AUC: 0.0957


/usr/local/lib/python3.11/dist-packages/pyro/poutine/lift_messenger.py:74: UserWarning: pyro.module prior did not find params ['module$$$fc2.weight', 'module$$$out.weight', 'module$$$fc2.bias', 'module$$$out.bias', 'module$$$fc1.weight', 'module$$$fc1.bias']. Did you instead mean one of ['fc1.bias', 'fc2.bias', 'fc1.weight', 'out.weight', 'out.bias', 'fc2.weight']?
  warnings.warn(


In [ ]:
best_thresh = 0.5
best_f1 = 0.0
for t in np.linspace(0, 1, 101):
    y_pred_temp = (y_prob_mean >= t).astype(int)
    f1_temp = f1_score(y_test_np, y_pred_temp)
    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_thresh = t

print(f"Best threshold: {best_thresh:.2f} with F1={best_f1:.4f}")
y_pred_class = (y_prob_mean >= best_thresh).astype(int)

acc = accuracy_score(y_test_np, y_pred_class)
print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test_np, y_pred_class))


Best threshold: 0.58 with F1=0.1521
Accuracy: 0.5579
              precision    recall  f1-score   support

         0.0       0.95      0.56      0.70     12574
         1.0       0.09      0.60      0.15       892

    accuracy                           0.56     13466
   macro avg       0.52      0.58      0.43     13466
weighted avg       0.89      0.56      0.66     13466

